# PEFT+AFSP — the adapter under retrieval prompts (val)

---
## 1 — Host, working tree, disk

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv

In [ ]:
from pathlib import Path

if not Path('manage.py').exists():
    if not Path('Style-Aware-MT/manage.py').exists():
        !git clone https://github.com/prnamhr/Style-Aware-MT.git
    %cd Style-Aware-MT
!git pull --ff-only
!git rev-parse --short HEAD

In [ ]:
!git log --oneline -1 -- docs/DEVLOG.md

In [ ]:
# %pip installs into the kernel; !pip may not.
%pip install -q -r requirements.txt

In [ ]:
# torch 2.12 breaks the pinned-torch ABI these three ship against; the pipeline is text-only.
%pip uninstall -q -y torchvision torchaudio torchcodec

In [ ]:
import torch

cap = torch.cuda.get_device_capability(0)
print(f'{torch.cuda.get_device_name(0)}  sm_{cap[0]}{cap[1]}',
      f'torch {torch.__version__} / cuda {torch.version.cuda}')
assert torch.cuda.is_bf16_supported(), 'bf16 unsupported; the frozen base is not quantized'

---
## 2 — Run parameters


In [ ]:
import json
import os
import time
from datetime import datetime, timedelta, timezone

import yaml

CONFIG = Path('configs/peft_afsp.yaml')
CONDITIONS = ['peft_knn', 'peft_afsp']
SPLIT = 'val'

CFG = yaml.safe_load(CONFIG.read_text(encoding='utf-8'))
GEN, RETR, AFSP = CFG['generator'], CFG['retrieval'], CFG['afsp']

assert GEN['model'] == 'Qwen/Qwen2.5-7B-Instruct', GEN['model']
assert (GEN['temperature'], GEN['top_p']) == (0.0, 1.0), 'not the locked greedy decoding'
assert (GEN['max_tokens'], GEN['seed']) == (1024, 42), GEN
assert GEN['dtype'] == 'bfloat16' and GEN['load_in_4bit'] is False, 'quantizing redefines the base'
assert GEN['adapter_path'] == 'models/peft_lora_r32_lr2e-4/checkpoint-1358', GEN['adapter_path']

# The frozen AFSP operating point (DEVLOG 2026-07-23). Nothing here is re-selected.
assert RETR['k'] == 8 and AFSP['beta'] == 0.3 and AFSP['lambda_style'] == 0.75, (RETR, AFSP)

EVAL_FILE = Path(CFG['data']['eval_file'])
assert EVAL_FILE.name == 'val.jsonl', 'the test split is sealed'
ROWS = [json.loads(x) for x in EVAL_FILE.open(encoding='utf-8') if x.strip()]
assert len(ROWS) == 1323, f'{len(ROWS)} segments, expected 1323'

BUDGET_H = 4
DEADLINE = datetime.now(timezone.utc) + timedelta(hours=BUDGET_H)
print(f'{len(CONDITIONS)} conditions x {len(ROWS)} segments, deadline {DEADLINE:%Y-%m-%d %H:%M}Z')

In [ ]:
import getpass
import logging

# HF_TOKEN only. The adapter repo is private and the base is public; nothing in this
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF_TOKEN: ')
for var in ('OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY'):
    assert not os.environ.get(var), f'{var} is set; this session makes no paid call'
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
logging.getLogger('httpx').setLevel(logging.WARNING)
print('HF_TOKEN set, no rater keys present')

---
## 3 — Weights and the index, before the GPU is touched

Network first. Every download that happens after the base is resident is a GPU-hour
spent waiting on bandwidth.

In [ ]:
import hashlib

from huggingface_hub import snapshot_download

HF_REPO = 'prnamhr/style-aware-mt-models'
ADAPTER = Path(GEN['adapter_path'])
ADAPTER_FILES = ('adapter_config.json', 'adapter_model.safetensors')

if not all((ADAPTER / f).exists() for f in ADAPTER_FILES):
    snapshot_download(HF_REPO, local_dir='.', token=os.environ['HF_TOKEN'],
                      allow_patterns=[f'{ADAPTER}/{f}' for f in ADAPTER_FILES])

conf = json.loads((ADAPTER / 'adapter_config.json').read_text(encoding='utf-8'))
assert conf['r'] == 32 and conf['lora_alpha'] == 64, conf
assert conf['base_model_name_or_path'].endswith(GEN['model'].split('/')[-1]), conf
ADAPTER_SHA = hashlib.sha256((ADAPTER / 'adapter_model.safetensors').read_bytes()).hexdigest()
print(f'{ADAPTER}  r={conf["r"]} alpha={conf["lora_alpha"]}  {ADAPTER_SHA[:12]}')

In [ ]:
# The base too, so section 5 loads from disk. ~15 GB in bf16 safetensors.
t0 = time.perf_counter()
snapshot_download(GEN['model'], allow_patterns=['*.json', '*.safetensors', '*.txt', '*.jinja'],
                  token=os.environ['HF_TOKEN'], max_workers=8)
print(f"{GEN['model']} cached in {(time.perf_counter() - t0) / 60:.1f} min")

In [ ]:
INDEX = Path(RETR['index_dir'])
INDEX_FILES = ('embeddings.npy', 'pairs.jsonl', 'meta.json')
INDEX_REBUILT = not all((INDEX / f).exists() for f in INDEX_FILES)

if INDEX_REBUILT:
    print('index missing -- rebuilding; exemplar selection may differ from the reported AFSP run')
    !python3 manage.py build_index --config {CONFIG}

INDEX_SHA = {f: hashlib.sha256((INDEX / f).read_bytes()).hexdigest() for f in INDEX_FILES}
meta = json.loads((INDEX / 'meta.json').read_text(encoding='utf-8'))
assert meta['embed_model'] == RETR['embed_model'] and meta['indexed_side'] == 'source', meta
assert meta['n_passages'] == 10860, meta
print(f'index {"rebuilt" if INDEX_REBUILT else "transferred"}, {meta["n_passages"]} passages')
for f, digest in INDEX_SHA.items():
    print(f'  {f:16s} {digest[:12]}')

---
## 4 — The gate

In [ ]:
from src.infer.run import _load_configured_glossary, build_fewshot_user, make_client, order_exemplars
from src.retrieval.retrieve import RetrievalIndex

index = RetrievalIndex(RETR['index_dir'], embed_model=RETR['embed_model'])
STYLE = Path(CFG['prompt']['style_instruction_file']).read_text(encoding='utf-8')
GLOSSARY = _load_configured_glossary(CFG)

PROBE_N = 8
probe_src = [r['input'] for r in ROWS[:PROBE_N]]
probe = [build_fewshot_user(s, order_exemplars(ex, CFG['prompt']['ordering']), GLOSSARY)
         for s, ex in zip(probe_src, index.retrieve(probe_src, k=RETR['k']))]

t0 = time.perf_counter()
client = make_client(GEN)
load_s = time.perf_counter() - t0

t0 = time.perf_counter()
for user in probe:
    client.complete(STYLE, user)
seg_s = (time.perf_counter() - t0) / PROBE_N

print(f'{load_s:.0f}s adapter+base load, {seg_s:.2f}s per segment at k={RETR["k"]}')
print(f'{torch.cuda.max_memory_reserved() / 2**30:.1f} GiB reserved')

In [ ]:
pass_h = len(ROWS) * seg_s / 3600
left_h = (DEADLINE - datetime.now(timezone.utc)).total_seconds() / 3600
print(f'{pass_h:.2f} h per pass, {pass_h * len(CONDITIONS):.1f} h for {len(CONDITIONS)}, '
      f'{left_h:.1f} h left of the booking')

if pass_h * len(CONDITIONS) > 0.9 * left_h:
    print('\nDoes not fit. Both passes are resumable through their own outputs/*_val.jsonl,')
    print('so the honest lever is more hours, not fewer segments: a capped pass cannot be')
    print('paired against outputs/peft_val.jsonl and scores nothing.')
else:
    print('\nFits. Section 5 may start.')

In [ ]:
# The probe held the base and the adapter; free them before manage.py loads its own.
del client, index
torch.cuda.empty_cache()

---
## 5 — Generation

In [ ]:
import subprocess
import sys

TIMING = {}
for cond in CONDITIONS:
    t0 = time.perf_counter()
    r = subprocess.run([sys.executable, 'manage.py', 'infer', '--condition', cond,
                        '--config', str(CONFIG)], check=False)
    assert r.returncode == 0, f'{cond} exited {r.returncode}'
    TIMING[cond] = {'seconds': round(time.perf_counter() - t0, 1),
                    'finished': datetime.now(timezone.utc).isoformat()}
    print(f'{cond}: {TIMING[cond]["seconds"] / 60:.1f} min')

In [ ]:
# Every segment present, aligned to val and to the reference the scoring pairs against.
VAL_SRC = [r['input'] for r in ROWS]
PEFT = Path('outputs/peft_val.jsonl')

for cond in CONDITIONS:
    rows = [json.loads(x) for x in Path(f'outputs/{cond}_{SPLIT}.jsonl').open(encoding='utf-8') if x.strip()]
    assert len(rows) == len(ROWS), f'{cond}: {len(rows)} rows, expected {len(ROWS)}'
    assert [r['input'] for r in rows] == VAL_SRC, f'{cond}: source order differs from val.jsonl'
    assert all(r['condition'] == cond for r in rows), f'{cond}: mislabelled rows'
    blank = [i for i, r in enumerate(rows) if not r['prediction'].strip()]
    print(f'{cond}: {len(rows)} rows, {len(blank)} blank {blank[:5]}')

if PEFT.exists():
    peft_src = [json.loads(x)['input'] for x in PEFT.open(encoding='utf-8') if x.strip()]
    assert peft_src == VAL_SRC, 'outputs/peft_val.jsonl does not pair with val.jsonl'
    print('reference outputs/peft_val.jsonl aligns')

---
## 6 — Manifest

In [ ]:
import platform

import peft as peft_lib
import transformers

MANIFEST = {
    'conditions': CONDITIONS,
    'split': SPLIT,
    'config': str(CONFIG),
    'base': GEN['model'],
    'generator': GEN,
    'retrieval': {'k': RETR['k'], 'embed_model': RETR['embed_model'], 'index_dir': str(INDEX)},
    'afsp': {k: AFSP[k] for k in ('beta', 'lambda_style', 'style_objective',
                                  'style_target_sigma', 'pool_mult', 'knn_hubness')},
    'adapter': {'path': str(ADAPTER), 'sha256': ADAPTER_SHA,
                'r': conf['r'], 'lora_alpha': conf['lora_alpha']},
    'index': {'rebuilt_here': INDEX_REBUILT, 'sha256': INDEX_SHA, 'meta': meta},
    'timing': TIMING,
    'commit': subprocess.run(['git', 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip(),
    'versions': {
        'device': torch.cuda.get_device_name(0),
        'torch': torch.__version__,
        'cuda': torch.version.cuda,
        'transformers': transformers.__version__,
        'peft': peft_lib.__version__,
        'python': platform.python_version(),
    },
}
out = Path('outputs/peft_afsp_manifest.json')
out.write_text(json.dumps(MANIFEST, indent=2) + '\n', encoding='utf-8')
print(json.dumps(MANIFEST, indent=2))

---
## 7 — Seal

In [ ]:
# 1. Nothing was spent.
for cond in CONDITIONS:
    usage = json.loads(Path(f'outputs/{cond}_{SPLIT}_usage.json').read_text(encoding='utf-8'))
    assert usage.get('cost_usd', 0.0) == 0.0, usage
    assert usage['provenance']['adapter_path'] == GEN['adapter_path'], usage['provenance']
    print(f'{cond}: {usage["calls"]} calls, ${usage.get("cost_usd", 0.0):.2f}, '
          f'provenance {json.dumps(usage["provenance"])}')

# 2. The test split was not touched.
assert not list(Path('outputs').glob('*_test.jsonl')), 'a test-split output exists'
assert not list(Path('results').glob('*_test.json')), 'a test-split result exists'

# 3. No rater key was ever present in this session.
for var in ('OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'GEMINI_API_KEY'):
    assert not os.environ.get(var), var
print('\nsealed: 0 paid calls, test split untouched, no rater key present')

In [ ]:
!tar -czf peft_afsp_val.tar.gz outputs/peft_knn_val.jsonl outputs/peft_afsp_val.jsonl \
    outputs/peft_knn_val_usage.json outputs/peft_afsp_val_usage.json outputs/peft_afsp_manifest.json
!ls -la peft_afsp_val.tar.gz
!git status --short outputs results